In [ ]:
# 1. IMPORT LIBRARIES, CONFIGURE PATHS & ARTIFACT SETUP
# ------------------------------------------------------------

import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

# ------------------------------------------------
# STREAMINTEL 360: PROJECT PATH SETUP
# ------------------------------------------------
# Runs locally in VS Code, reading already-trained artifacts from
# Notebooks 01 (churn) and 03 (recommendations).
PROJECT_ROOT = Path(r"E:\STREAMINTEL360_Complete")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

ARTIFACT_ROOT = ARTIFACTS_DIR / "notebook_08_explainable_ai"
REPORTS_DIR = ARTIFACT_ROOT / "reports"
METRICS_DIR = ARTIFACT_ROOT / "metrics"

for dir_path in [REPORTS_DIR, METRICS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------
# SOURCE ARTIFACT DIRECTORIES
# ------------------------------------------------
CHURN_DIR = ARTIFACTS_DIR / "notebook_01_churn"
RECOMMENDATION_DIR = ARTIFACTS_DIR / "notebook_03_recommendations"

PROJECT_NAME = "STREAMINTEL 360"
MODULE_NAME = "Explainable AI"

print("STREAMINTEL 360 — EXPLAINABLE AI ENVIRONMENT INITIALIZATION")
print(f"Project       : {PROJECT_NAME}")
print(f"Module        : {MODULE_NAME}")
print(f"Artifacts Dir : {ARTIFACTS_DIR.resolve()}")
print(f"Artifact Root : {ARTIFACT_ROOT.resolve()}")

print("\nSOURCE ARTIFACT DIRECTORIES")
for name, path in {"Churn": CHURN_DIR, "Recommendation": RECOMMENDATION_DIR}.items():
    print(f"• {name:<15}: {'FOUND' if path.exists() else 'NOT FOUND'} ({path})")


In [ ]:
# 2. LOAD EXISTING CHURN ARTIFACTS
# ------------------------------------------------------------

CHURN_MODEL_PATH = CHURN_DIR / "models" / "best_model.joblib"
CHURN_SCALER_PATH = CHURN_DIR / "preprocessors" / "scaler.joblib"
CHURN_FEATURES_PATH = CHURN_DIR / "preprocessors" / "feature_columns.joblib"

required_files = {
    "Churn Model": CHURN_MODEL_PATH,
    "Scaler": CHURN_SCALER_PATH,
    "Feature Columns": CHURN_FEATURES_PATH,
}

missing_files = [name for name, path in required_files.items() if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing required churn artifacts:\n"
        + "\n".join(f"• {name}: {required_files[name]}" for name in missing_files)
    )

churn_model = joblib.load(CHURN_MODEL_PATH)
churn_scaler = joblib.load(CHURN_SCALER_PATH)
churn_feature_columns = joblib.load(CHURN_FEATURES_PATH)

if isinstance(churn_feature_columns, np.ndarray):
    churn_feature_columns = churn_feature_columns.tolist()
churn_feature_columns = list(churn_feature_columns)

# This explainability method (coefficient x scaled value) requires a
# linear model. Notebook 01 auto-selects its best model by F1 score,
# which is not guaranteed to be Logistic Regression on every run.
if not hasattr(churn_model, "coef_"):
    raise TypeError(
        f"Loaded churn model is '{type(churn_model).__name__}', which has "
        "no 'coef_' attribute. This notebook requires a linear model "
        "(e.g. Logistic Regression) for coefficient-based explanations."
    )

print("CHURN ARTIFACTS LOADED SUCCESSFULLY")
print(f"• Model           : {type(churn_model).__name__}")
print(f"• Scaler          : {type(churn_scaler).__name__}")
print(f"• Feature Columns : {len(churn_feature_columns)}")
print("\nArtifact Validation: PASSED")


CHURN ARTIFACTS LOADED SUCCESSFULLY
• Model           : LogisticRegression
• Scaler          : StandardScaler
• Feature Columns : 30

Artifact Validation: PASSED


c:\Users\hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.6.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
# 3. CHURN PREDICTION + FEATURE CONTRIBUTIONS
# ------------------------------------------------------------

# Example subscriber profile for explanation
subscriber_profile = {
    "SeniorCitizen": 0, "tenure_months": 8, "monthly_fee_usd": 85.0, "total_spend_usd": 680.0,
    "gender_Male": 1, "Partner_Yes": 0, "Dependents_Yes": 0, "PhoneService_Yes": 1,
    "MultipleLines_No phone service": 0, "MultipleLines_Yes": 1,
    "InternetService_Fiber optic": 1, "InternetService_No": 0,
    "OnlineSecurity_No internet service": 0, "OnlineSecurity_Yes": 0,
    "OnlineBackup_No internet service": 0, "OnlineBackup_Yes": 0,
    "DeviceProtection_No internet service": 0, "DeviceProtection_Yes": 0,
    "streaming_support_tickets_No internet service": 0, "streaming_support_tickets_Yes": 1,
    "StreamingTV_No internet service": 0, "StreamingTV_Yes": 1,
    "StreamingMovies_No internet service": 0, "StreamingMovies_Yes": 1,
    "subscription_tier_One year": 0, "subscription_tier_Two year": 0,
    "PaperlessBilling_Yes": 1, "PaymentMethod_Credit card (automatic)": 0,
    "PaymentMethod_Electronic check": 1, "PaymentMethod_Mailed check": 0,
}

# Build a full feature row -- missing columns (features not in the
# hand-written profile above) default to 0
row = {col: subscriber_profile.get(col, 0) for col in churn_feature_columns}
subscriber_df = pd.DataFrame([row], columns=churn_feature_columns)
subscriber_scaled = churn_scaler.transform(subscriber_df)

churn_probability = churn_model.predict_proba(subscriber_scaled)[0, 1]
churn_prediction = int(churn_probability >= 0.5)

coefficients = churn_model.coef_[0]
contributions = subscriber_scaled[0] * coefficients

explanation_df = pd.DataFrame({
    "feature": churn_feature_columns,
    "feature_value": subscriber_df.iloc[0].values,
    "scaled_value": subscriber_scaled[0],
    "coefficient": coefficients,
    "contribution": contributions,
})
explanation_df["absolute_contribution"] = explanation_df["contribution"].abs()
explanation_df = explanation_df.sort_values("absolute_contribution", ascending=False).reset_index(drop=True)

prediction_label = "HIGHER CHURN RISK" if churn_prediction == 1 else "LOWER CHURN RISK"

print("PREDICTION")
print(f"• Churn Probability : {churn_probability:.4f}")
print(f"• Prediction        : {prediction_label}")

print("\nTOP 10 FEATURE CONTRIBUTIONS")
display(explanation_df[["feature", "feature_value", "coefficient", "contribution"]].head(10))

print("\nExplainability Method:")
print("• Logistic Regression coefficient × scaled feature value")
print("• Positive contribution → pushes prediction toward churn")
print("• Negative contribution → pushes prediction away from churn")


PREDICTION
• Churn Probability : 0.7948
• Prediction        : HIGHER CHURN RISK

TOP 10 FEATURE CONTRIBUTIONS


,feature,feature_value,coefficient,contribution
0,tenure_months,8.0,-1.347613,1.348825
1,InternetService_Fiber optic,1.0,0.727745,0.818642
2,monthly_fee_usd,85.0,-0.851551,-0.565722
3,total_spend_usd,680.0,0.639028,-0.455484
4,subscription_tier_Two year,0.0,-0.602591,0.339947
5,StreamingTV_Yes,1.0,0.249702,0.312959
6,StreamingMovies_Yes,1.0,0.236368,0.295474
7,PaymentMethod_Electronic check,1.0,0.181473,0.253391
8,MultipleLines_Yes,1.0,0.214359,0.249845
9,streaming_support_tickets_Yes,1.0,-0.118240,-0.182896



Explainability Method:
• Logistic Regression coefficient × scaled feature value
• Positive contribution → pushes prediction toward churn
• Negative contribution → pushes prediction away from churn


In [ ]:
# 4. CHURN EXPLANATION SUMMARY
# ------------------------------------------------------------

positive_drivers = explanation_df[explanation_df["contribution"] > 0].sort_values("contribution", ascending=False).head(5).copy()
negative_drivers = explanation_df[explanation_df["contribution"] < 0].sort_values("contribution", ascending=True).head(5).copy()

explanation_summary = {
    "prediction": prediction_label,
    "churn_probability": round(float(churn_probability), 4),
    "positive_churn_drivers": positive_drivers[["feature", "feature_value", "contribution"]].to_dict(orient="records"),
    "negative_churn_drivers": negative_drivers[["feature", "feature_value", "contribution"]].to_dict(orient="records"),
}

print("TOP FACTORS PUSHING THE MODEL TOWARD CHURN")
for row in positive_drivers.itertuples(index=False):
    print(f"• {row.feature}: contribution = {row.contribution:+.4f}")

print("\nTOP FACTORS PUSHING THE MODEL AWAY FROM CHURN")
for row in negative_drivers.itertuples(index=False):
    print(f"• {row.feature}: contribution = {row.contribution:+.4f}")

explanation_file = REPORTS_DIR / "churn_explanation.json"
with open(explanation_file, "w", encoding="utf-8") as f:
    json.dump(explanation_summary, f, indent=2, ensure_ascii=False)

print(f"\nChurn explanation saved to: {explanation_file}")


TOP FACTORS PUSHING THE MODEL TOWARD CHURN
• tenure_months: contribution = +1.3488
• InternetService_Fiber optic: contribution = +0.8186
• subscription_tier_Two year: contribution = +0.3399
• StreamingTV_Yes: contribution = +0.3130
• StreamingMovies_Yes: contribution = +0.2955

TOP FACTORS PUSHING THE MODEL AWAY FROM CHURN
• monthly_fee_usd: contribution = -0.5657
• total_spend_usd: contribution = -0.4555
• streaming_support_tickets_Yes: contribution = -0.1829
• DeviceProtection_Yes: contribution = -0.0501
• SeniorCitizen: contribution = -0.0311

Churn explanation saved to: E:\STREAMINTEL360_Complete\artifacts\notebook_08_explainable_ai\reports\churn_explanation.json


In [ ]:
# 5. GLOBAL CHURN FEATURE IMPORTANCE
# ------------------------------------------------------------

global_importance_df = pd.DataFrame({
    "feature": churn_feature_columns,
    "coefficient": churn_model.coef_[0],
})
global_importance_df["importance"] = global_importance_df["coefficient"].abs()
global_importance_df["direction"] = np.where(global_importance_df["coefficient"] > 0, "Toward churn", "Away from churn")
global_importance_df = global_importance_df.sort_values("importance", ascending=False).reset_index(drop=True)

print("TOP 10 GLOBAL FEATURES")
display(global_importance_df[["feature", "coefficient", "importance", "direction"]].head(10))

global_importance_file = METRICS_DIR / "churn_global_feature_importance.csv"
global_importance_df.to_csv(global_importance_file, index=False)

print(f"\nGlobal feature importance saved to: {global_importance_file}")


TOP 10 GLOBAL FEATURES


,feature,coefficient,importance,direction
0,tenure_months,-1.347613,1.347613,Away from churn
1,monthly_fee_usd,-0.851551,0.851551,Away from churn
2,InternetService_Fiber optic,0.727745,0.727745,Toward churn
3,total_spend_usd,0.639028,0.639028,Toward churn
4,subscription_tier_Two year,-0.602591,0.602591,Away from churn
5,subscription_tier_One year,-0.310898,0.310898,Away from churn
6,StreamingTV_Yes,0.249702,0.249702,Toward churn
7,StreamingMovies_Yes,0.236368,0.236368,Toward churn
8,MultipleLines_Yes,0.214359,0.214359,Toward churn
9,PaymentMethod_Electronic check,0.181473,0.181473,Toward churn



Global feature importance saved to: E:\STREAMINTEL360_Complete\artifacts\notebook_08_explainable_ai\metrics\churn_global_feature_importance.csv


In [ ]:
# 6. LOAD RECOMMENDATION EVIDENCE
# ------------------------------------------------------------

HYBRID_CONFIG_PATH = RECOMMENDATION_DIR / "models" / "hybrid_model_config.joblib"
MODEL_MANIFEST_PATH = RECOMMENDATION_DIR / "models" / "model_manifest.json"
HYBRID_METRICS_PATH = RECOMMENDATION_DIR / "metrics" / "hybrid_evaluation_metrics.json"

required_recommendation_files = {
    "Hybrid Model Config": HYBRID_CONFIG_PATH,
    "Model Manifest": MODEL_MANIFEST_PATH,
    "Hybrid Evaluation Metrics": HYBRID_METRICS_PATH,
}

missing_files = [name for name, path in required_recommendation_files.items() if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing recommendation artifacts:\n"
        + "\n".join(f"• {name}: {required_recommendation_files[name]}" for name in missing_files)
    )

# hybrid_model_config.joblib holds the DEFAULT weights hardcoded early
# in Notebook 03 (before grid-search optimization ran).
hybrid_config = joblib.load(HYBRID_CONFIG_PATH)

# model_manifest.json holds the ACTUAL data-driven optimal weights
# from the grid search.
with open(MODEL_MANIFEST_PATH, "r", encoding="utf-8") as f:
    model_manifest = json.load(f)

with open(HYBRID_METRICS_PATH, "r", encoding="utf-8") as f:
    hybrid_metrics = json.load(f)

print("RECOMMENDATION EVIDENCE LOADED")
print("\nDefault Hybrid Config (pre-optimization):")
print(json.dumps(hybrid_config, indent=2))
print("\nModel Manifest (optimal, post-grid-search):")
print(json.dumps(model_manifest, indent=2))
print("\nHybrid Evaluation Metrics:")
print(json.dumps(hybrid_metrics, indent=2))

print("\nRecommendation evidence validation: PASSED")


RECOMMENDATION EVIDENCE LOADED

Default Hybrid Config (pre-optimization):
{
  "weights": {
    "w_collab": 0.45,
    "w_content": 0.35,
    "w_pop": 0.2
  },
  "default_top_n": 10,
  "model_components": [
    "Item_CF",
    "TFIDF_Content",
    "Popularity_Baseline"
  ]
}

Model Manifest (optimal, post-grid-search):
{
  "model_name": "StreamIntelRecommenderEngine",
  "catalog_version": "1.0.0",
  "artifact_file": "recommendation_engine_v1.joblib",
  "compressed_size_mb": 11.41,
  "compression_level": 3,
  "optimal_weights": {
    "w_collab": 0.7,
    "w_content": 0.1,
    "w_pop": 0.2,
    "optimal_map_at_10": 0.005
  },
  "evaluation_metrics": {
    "k": 10,
    "map_at_k": 0.005
  }
}

Hybrid Evaluation Metrics:
{
  "model_evaluated": "Hybrid_Recommendation_Engine",
  "evaluated_users_count": 300,
  "cutoff_k": 10,
  "precision_at_k": 0.0023,
  "recall_at_k": 0.0111,
  "map_at_k": 0.0045,
  "ndcg_at_k": 0.0068,
  "hit_rate_at_k": 0.0233
}

Recommendation evidence validation: PASSED


In [ ]:
# 7. RECOMMENDATION EXPLANATION + FINAL XAI REPORT
# ------------------------------------------------------------

default_weights = hybrid_config.get("weights", {})
optimal_weights = model_manifest.get("optimal_weights", {})

# Real comparison: the config saved BEFORE grid-search optimization
# vs. the weights the search actually found to be best. These can
# genuinely differ since hybrid_config.joblib was hardcoded early.
weights_match = all(
    round(default_weights.get(k, -1), 2) == round(optimal_weights.get(k, -2), 2)
    for k in ("w_collab", "w_content", "w_pop")
)

recommendation_explanation = {
    "explainability_method": "Evidence-based hybrid recommendation weight comparison",
    "model_components": hybrid_config.get("model_components", []),
    "default_weights_pre_optimization": default_weights,
    "optimal_weights_post_grid_search": optimal_weights,
    "weights_match": weights_match,
    "reported_evaluation_metrics": hybrid_metrics,
    "interpretation": {
        "configuration_note": (
            "The hardcoded default weights match the grid-search optimal "
            "weights -- the initial configuration happened to already be optimal."
            if weights_match else
            "The hardcoded default weights differ from the grid-search optimal "
            "weights. The production engine (recommendation_engine_v1.joblib) "
            "uses the optimal weights, not the early default."
        ),
        "performance": (
            "The reported ranking metrics reflect the actual evaluation run "
            "and should be read as current evidence, not a guarantee of "
            "strong production performance."
        ),
    },
}

recommendation_explanation_file = REPORTS_DIR / "recommendation_explanation.json"
with open(recommendation_explanation_file, "w", encoding="utf-8") as f:
    json.dump(recommendation_explanation, f, indent=2, ensure_ascii=False)

# Final XAI Summary
xai_report = {
    "project": PROJECT_NAME,
    "module": MODULE_NAME,
    "explainability_modules": {
        "churn": {
            "status": "Completed",
            "method": "Logistic Regression coefficient × scaled feature value",
            "prediction_probability": round(float(churn_probability), 4),
            "prediction": prediction_label,
            "artifact": str(explanation_file),
            "global_feature_importance": str(global_importance_file),
        },
        "recommendation": {
            "status": "Completed",
            "method": "Default vs. optimal hybrid weight comparison",
            "weights_match": weights_match,
            "artifact": str(recommendation_explanation_file),
        },
    },
    "evidence_policy": "Explanations are derived from existing model artifacts and evaluation results. No model was retrained.",
}

xai_report_file = REPORTS_DIR / "explainability_report.json"
with open(xai_report_file, "w", encoding="utf-8") as f:
    json.dump(xai_report, f, indent=2, ensure_ascii=False)

print("XAI ARTIFACTS GENERATED")
print(f"• Churn Explanation   : {explanation_file}")
print(f"• Global Feature CSV  : {global_importance_file}")
print(f"• Recommendation XAI  : {recommendation_explanation_file}")
print(f"• Final XAI Report    : {xai_report_file}")

print("\nRECOMMENDATION WEIGHTS COMPARISON")
print(f"• Default  (pre-optimization) : {default_weights}")
print(f"• Optimal  (grid search)      : {optimal_weights}")
print(f"• Weights Match                : {weights_match}")

print("\n" + "=" * 70)
print("STREAMINTEL 360 — NOTEBOOK 08 COMPLETE")
print("=" * 70)


XAI ARTIFACTS GENERATED
• Churn Explanation   : E:\STREAMINTEL360_Complete\artifacts\notebook_08_explainable_ai\reports\churn_explanation.json
• Global Feature CSV  : E:\STREAMINTEL360_Complete\artifacts\notebook_08_explainable_ai\metrics\churn_global_feature_importance.csv
• Recommendation XAI  : E:\STREAMINTEL360_Complete\artifacts\notebook_08_explainable_ai\reports\recommendation_explanation.json
• Final XAI Report    : E:\STREAMINTEL360_Complete\artifacts\notebook_08_explainable_ai\reports\explainability_report.json

RECOMMENDATION WEIGHTS COMPARISON
• Default  (pre-optimization) : {'w_collab': 0.45, 'w_content': 0.35, 'w_pop': 0.2}
• Optimal  (grid search)      : {'w_collab': 0.7, 'w_content': 0.1, 'w_pop': 0.2, 'optimal_map_at_10': 0.005}
• Weights Match                : False

STREAMINTEL 360 — NOTEBOOK 08 COMPLETE
